In [1]:
"""
This script is used to analyze the core losses in differently shaped ferrite cores.
"""
# %load_ext autoreload
# %autoreload 2
import numpy as np
import mph
import mphsweepkit as msk

from dimensional_performance_factor.meta import paths

# Paths:
material_path = paths.material_data
simulation_path = paths.simulation_data
plot_path = paths.grafics

Start the Comsol Server and load the model

In [2]:
# Start the COMSOL client
client = mph.start()

In [3]:
# Load the model
model = client.load(simulation_path.joinpath('frame-init.mph'))

Initialize the CascadedSweepModel

In [4]:
# Data is stored in data_dir in global_data/, field_data/, and batch_data/.
csm = msk.CascadedSweepModel(
    model,
    'Frame Study',
    show_param_names=True,
    data_dir=simulation_path,
)

Initialized CascadedSweepModel
Study name: Frame Study
Sweep Structure:
    - Geometry Sweep (BatchSweep) -> 'cut_switch'
      - Material Sweep (MaterialSweep) -> 'matsw.comp1.sw1'
        - Excitation Sweep (Parametric) -> 'b_mean', 'core_temperature'
          - Coil Geometry Analysis (CoilCurrentCalculation)
            - Frequency Sweep (Frequency)
Loop names: ['Geometry Sweep', 'Material Sweep', 'Excitation Sweep', 'Coil Geometry Analysis', 'Frequency Sweep']
Loop lengths: [2, 2, 1, 1, 12]
Added 'geometry_idx' and 'internal_idx' columns to input_data. New shape: (48, 7)
--------------------------------
Data updated from MPh-model.
Input data shape: (48, 7)
Reset output data to shape of the input data: (48, 1)
Combined shape: (48, 8)


In [5]:
# Materials inspection
material_overview = csm.get_material_overview()
print(material_overview, "\n")



# Set the material paths to the interpolation functions for N49 (LEA_MTB)
csm.set_material_function_path("N49 (LEA_MTB)", "MagneticLosses", material_path, "N49_LEA_MTB_permeability_grid.txt")
csm.set_material_function_path("N49 (LEA_MTB)", "DielectricLoss", material_path, "N49_LEA_MTB_permittivity_grid.txt")

# Set the material function path to the interpolation function for N49 static (LEA_MTB)
csm.set_material_function_path("N49 static (LEA_MTB)", "MagneticLosses", material_path, "N49_LEA_MTB_permeability_grid.txt")

                   name   tag      type parent_tag
0                Copper  mat1     solid       None
1                   Air  mat2  nonSolid       None
2     Material Switch 1   sw1     solid       None
3         N49 (LEA_MTB)  mat4     solid        sw1
4  N49 static (LEA_MTB)  mat5     solid        sw1
5        Linear_Ferrite  mat3     solid       None 



In [6]:
csm.set_parametric_sweep(sweep_name='Excitation Sweep',
                         param_names=["b_mean", "core_temperature"],
                         param_units=["T", "C"],
                         param_values=[[0.05], [30.0, 60.0, 90.0]],
                         sweep_type="filled")

Loop names: ['Geometry Sweep', 'Material Sweep', 'Excitation Sweep', 'Coil Geometry Analysis', 'Frequency Sweep']
Loop lengths: [2, 2, 3, 1, 12]
Added 'geometry_idx' and 'internal_idx' columns to input_data. New shape: (144, 7)
--------------------------------
Data updated from MPh-model.
Input data shape: (144, 7)
Reset output data to shape of the input data: (144, 1)
Combined shape: (144, 8)
Added 'geometry_idx' and 'internal_idx' columns to input_data. New shape: (144, 7)


In [7]:
csm.save_global_data()

Saved result data to: \\fs-cifs\upb\groups\lea\data\Employees\tpiepe\data_frame\simulation_data\global_data


In [8]:
# TODO: allow to change the mesh and set the accuracy
csm.simulate()

Batch directory for the geometric sweep set to: N:\groups\lea\data\Employees\tpiepe\data_frame\simulation_data\batch_data


In [9]:
# Save to new file
model.save(simulation_path.joinpath('frame-solved.mph'))

In [10]:
client.remove(model)

In [11]:
client.clear()
client.disconnect()